# 01. Connect to the Exchange and Inventory Available Data

This notebook enumerates every study, patient, consent scope, and observation type visible to the logged-in account, then writes an aggregate snapshot to `outputs/inventory_snapshot.csv`. The snapshot contains counts only and is safe to commit.

Run it on the JupyterHealth Hub, where `JHE_URL` and `JHE_TOKEN` are already set.

In [ ]:
import pandas as pd
from jupyterhealth_client import JupyterHealthClient, Code

client = JupyterHealthClient()
client.get_user()

## Studies

In [ ]:
studies = pd.DataFrame([
    {"study_id": s["id"], "study_name": s["name"], "organization": s["organization"]["name"]}
    for s in client.list_studies()
])
studies

## Patients and consented scopes

One row per patient, study, and consented measurement type. Consent is granted per study and per scope, so this table shows exactly which data each study is permitted to see.

In [ ]:
rows = []
for s in studies.itertuples():
    for p in client.list_patients(study_id=s.study_id):
        consents = client.get_patient_consents(p["id"])
        for study in consents["studies"]:
            if study["id"] != s.study_id:
                continue
            for scope in study["scopeConsents"]:
                rows.append({
                    "study_id": s.study_id,
                    "patient_id": p["id"],
                    "scope": scope["code"]["codingCode"],
                    "consented": scope["consented"],
                })

consents = pd.DataFrame(rows)
consents.groupby(["study_id", "scope", "consented"]).patient_id.nunique().unstack(fill_value=0)

## Observation counts by study and type

`limit=None` retrieves every record, which can be slow for large studies. Reduce it if a study is very large.

In [ ]:
summary = []
for s in studies.itertuples():
    for code in Code:
        df = client.list_observations_df(study_id=s.study_id, code=code, limit=None)
        if len(df) == 0:
            continue
        t = df["effective_time_frame_date_time"]
        summary.append({
            "study_id": s.study_id,
            "study_name": s.study_name,
            "code": code.value,
            "n_patients": df["subject_reference"].nunique(),
            "n_observations": len(df),
            "first": t.min().date(),
            "last": t.max().date(),
        })

summary = pd.DataFrame(summary)
summary

In [ ]:
summary.to_csv("../outputs/inventory_snapshot.csv", index=False)
print(summary.to_markdown(index=False))  # paste into docs/02_data-inventory.md

## What else the Exchange exposes

These answer the open questions in `docs/02_data-inventory.md`: which FHIR resource types exist beyond Observation, and which data sources are registered.

In [ ]:
client.list_fhir_resource_types()

In [ ]:
pd.DataFrame(list(client.list_data_sources()))